In [ ]:
import requests
import time

url = "https://magma.esdm.go.id/v1/vona/"

# Retry logic untuk mengatasi masalah koneksi
max_retries = 3
retry_delay = 2  # detik

for attempt in range(max_retries):
    try:
        print(f"Mencoba koneksi... (Percobaan {attempt + 1}/{max_retries})")
        
        # Melakukan request HTTP dengan timeout
        response = requests.get(url, timeout=15)
        
        if response.status_code == 200:
            print("✓ Koneksi Berhasil! Status: 200 OK")
            print("Data siap di-scrape.")
            break
        else:
            print(f"Server merespon dengan kode: {response.status_code}")
            
    except requests.exceptions.ConnectionError as e:
        print(f"✗ Koneksi gagal: Tidak dapat terhubung ke server")
        print(f"  Error: {str(e)[:100]}...")
        if attempt < max_retries - 1:
            print(f"  Mencoba lagi dalam {retry_delay} detik...")
            time.sleep(retry_delay)
        else:
            print("\nSolusi yang bisa dicoba:")
            print("1. Periksa koneksi internet Anda")
            print("2. Coba buka URL di browser: https://magma.esdm.go.id/v1/vona/")
            print("3. Flush DNS cache: ipconfig /flushdns (CMD as admin)")
            print("4. Ganti DNS ke 8.8.8.8 (Google) atau 1.1.1.1 (Cloudflare)")
            
    except Exception as e:
        print(f"✗ Error tidak terduga: {type(e).__name__}")
        print(f"  Detail: {e}")
        if attempt < max_retries - 1:
            print(f"  Mencoba lagi dalam {retry_delay} detik...")
            time.sleep(retry_delay)

# 🌋 VONA Data Scraper untuk Volcanic Ash Predictor

## 📝 Deskripsi Proyek
Scraper ini mengumpulkan data **VONA (Volcano Observatory Notice for Aviation)** dari MAGMA Indonesia untuk mendukung proyek **Volcanic Ash Predictor** menggunakan Machine Learning.

## 🎯 Tujuan
1. **Data untuk Model AI**: Tinggi kolom abu, koordinat gunung, timestamp
2. **Data untuk Aplikasi**: Alert level, info cuaca, rekomendasi keamanan
3. **Integrasi dengan OpenWeatherMap**: Data cuaca real-time berdasarkan koordinat & waktu

## 📦 Data yang Dikumpulkan

### Data Dasar (dari halaman list):
- ⏰ Timestamp
- 🚦 Alert Level (Green/Yellow/Orange/Red)
- 🌋 Volcano Report
- 👤 Author & Observatory
- 📄 Description

### Data Detail (dari setiap link VONA):
- 🌋 **Volcano Name & Code**
- 📍 **Koordinat (Latitude, Longitude)** → untuk peta Leaflet.js
- 🏔️ **Summit Elevation (FT & M)** → untuk kalkulasi
- ✈️ **Volcanic Cloud Height** → **DATA PALING PENTING untuk AI model**
- 🌤️ **Remarks** → info cuaca & arah angin
- 📋 Notice Number, Area, Current/Previous Color Code

## 🚀 Cara Penggunaan
1. **Test Koneksi** → Jalankan cell pertama
2. **Scraping Cepat (Optional)** → Cell scraping basic (tanpa detail)
3. **Scraping Lengkap (Recommended)** → Cell scraping complete (dengan detail)
4. **Simpan ke CSV** → Cell save data
5. **Lihat Statistik** → Cell analisis

---

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_vona_data(pages=1):
    base_url = "https://magma.esdm.go.id/v1/vona/"
    all_data = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for page in range(1, pages + 1):
        try:
            print(f"Scraping halaman {page}...")
            
            # Mengambil halaman web dengan retry
            url = f"{base_url}?page={page}" if page > 1 else base_url
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"  Gagal mengakses halaman {page}: Status {response.status_code}")
                continue
                
            soup = BeautifulSoup(response.text, 'html.parser')

            # Mencari semua item VONA (berdasarkan struktur HTML yang sebenarnya)
            reports = soup.find_all('div', class_='timeline-item')

            for report in reports:
                try:
                    # Skip timeline-day items (header pemisah tanggal)
                    if 'timeline-day' in report.get('class', []):
                        continue
                    
                    # Ekstrak waktu dan alert level
                    timeline_time = report.find('div', class_='timeline-time')
                    if not timeline_time:
                        continue
                    
                    timestamp_elem = timeline_time.find('small')
                    timestamp = timestamp_elem.get_text(strip=True) if timestamp_elem else 'N/A'
                    
                    # Ekstrak alert level (Green/Yellow/Orange/Red)
                    alert_button = timeline_time.find('a', class_='btn')
                    alert_level = alert_button.get_text(strip=True) if alert_button else 'N/A'
                    
                    # Ekstrak isi report
                    timeline_body = report.find('div', class_='timeline-body')
                    if not timeline_body:
                        continue
                    
                    # Nama gunung dan kode
                    title_elem = timeline_body.find('p', class_='timeline-title')
                    title = title_elem.get_text(strip=True) if title_elem else 'N/A'
                    
                    # Author dan observatory
                    author_elem = timeline_body.find('p', class_='timeline-author')
                    author_info = author_elem.get_text(strip=True) if author_elem else 'N/A'
                    
                    # Deskripsi aktivitas
                    text_elem = timeline_body.find('p', class_='timeline-text')
                    description = text_elem.get_text(strip=True) if text_elem else 'N/A'
                    
                    # Link detail
                    link_elem = timeline_body.find('a', class_='card-link')
                    detail_link = link_elem.get('href') if link_elem else 'N/A'
                    
                    all_data.append({
                        "Timestamp": timestamp,
                        "Alert_Level": alert_level,
                        "Volcano_Report": title,
                        "Author_Observatory": author_info,
                        "Description": description,
                        "Detail_Link": detail_link,
                        "Page": page
                    })
                    
                except Exception as e:
                    print(f"  Error parsing report: {e}")
                    continue
            
            print(f"  ✓ Halaman {page} selesai. Total data: {len(all_data)}")
            
            # Delay antar halaman untuk menghindari rate limiting
            if page < pages:
                time.sleep(1)
                
        except Exception as e:
            print(f"  ✗ Error pada halaman {page}: {e}")
            continue

    if not all_data:
        print("\n⚠ Tidak ada data yang berhasil di-scrape!")
        print("Kemungkinan masalah:")
        print("- Koneksi bermasalah")
        print("- Struktur HTML website berubah")
        print("- Perlu inspeksi ulang selector HTML")
        return pd.DataFrame()
    
    return pd.DataFrame(all_data)

# Contoh menjalankan untuk 2 halaman pertama (kurangi untuk testing)
try:
    df_vona = scrape_vona_data(pages=2)
    if not df_vona.empty:
        print(f"\n✓ Berhasil scrape {len(df_vona)} records")
        print(df_vona.head())
    else:
        print("\n✗ DataFrame kosong, scraping gagal")
except Exception as e:
    print(f"\n✗ Error utama: {e}")

## 📋 Scraping Data VONA - Mode Pilihan

Ada **2 mode scraping** yang tersedia:

### 1. **Scraping Cepat** (Hanya List - Cell di atas)
- ⚡ **Cepat** - hanya ambil data dari halaman list
- 📊 Data: Timestamp, Alert Level, Volcano Report, Author, Description
- ✅ **Gunakan untuk**: Preview cepat atau monitoring rutin

### 2. **Scraping Lengkap** (Dengan Detail - Cell di bawah) 
- 🔬 **Lengkap** - ambil detail dari setiap link VONA
- 📊 Data tambahan yang didapat:
  - ✈️ **Volcano Cloud Height** (tinggi kolom abu) - *untuk AI model*
  - 📍 **Koordinat** (Latitude, Longitude) - *untuk peta*
  - 🏔️ **Summit Elevation** (ketinggian puncak) - *untuk kalkulasi*
  - 🌤️ **Remarks** (info cuaca & arah angin) - *untuk prediksi*
  - 📋 Notice Number, Area, Volcano Code, dll.
- ⏱️ **Lebih lambat** - karena harus buka setiap detail page
- ✅ **Gunakan untuk**: Training model AI & aplikasi lengkap

---

### 💡 Rekomendasi untuk Proyek Volcanic Ash Predictor:
**Gunakan Mode Lengkap** karena Anda memerlukan:
1. **Tinggi kolom abu** → Input utama model AI
2. **Koordinat gunung** → Untuk peta interaktif
3. **Info cuaca/angin** → Untuk prediksi arah sebaran abu

In [ ]:
import re

def scrape_vona_detail(detail_url, headers):
    """
    Scrape informasi detail dari halaman VONA individual
    Mengambil data penting untuk model AI dan aplikasi
    """
    try:
        response = requests.get(detail_url, headers=headers, timeout=15)
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('table', class_='table-mg-b')
        
        if not table:
            return None
        
        # Inisialisasi data detail
        detail_data = {
            'Volcano_Name': 'N/A',
            'Volcano_Code': 'N/A',
            'Issued_Time': 'N/A',
            'Current_Color_Code': 'N/A',
            'Previous_Color_Code': 'N/A',
            'Notice_Number': 'N/A',
            'Volcano_Location': 'N/A',
            'Latitude': None,
            'Longitude': None,
            'Area': 'N/A',
            'Summit_Elevation_FT': None,
            'Summit_Elevation_M': None,
            'Volcanic_Activity_Summary': 'N/A',
            'Volcanic_Cloud_Height': 'N/A',
            'Other_Volcanic_Cloud_Info': 'N/A',
            'Remarks': 'N/A'
        }
        
        # Parse semua row dalam tabel
        rows = table.find_all('tr')
        
        for row in rows:
            cells = row.find_all('td')
            if len(cells) < 3:
                continue
            
            row_num = cells[0].get_text(strip=True)
            
            # (2) Issued Time
            if row_num == '(2)':
                detail_data['Issued_Time'] = cells[3].get_text(strip=True)
            
            # (3) Volcano Name and Code
            elif row_num == '(3)':
                volcano_text = cells[3].get_text(strip=True)
                # Parse "Kaba (261220)" -> nama: Kaba, kode: 261220
                match = re.match(r'(.+?)\s*\((\d+)\)', volcano_text)
                if match:
                    detail_data['Volcano_Name'] = match.group(1).strip()
                    detail_data['Volcano_Code'] = match.group(2).strip()
                else:
                    detail_data['Volcano_Name'] = volcano_text
            
            # (4) Current Aviation Colour Code
            elif row_num == '(4)':
                detail_data['Current_Color_Code'] = cells[3].get_text(strip=True)
            
            # (5) Previous Aviation Colour Code
            elif row_num == '(5)':
                detail_data['Previous_Color_Code'] = cells[3].get_text(strip=True)
            
            # (7) Notice Number
            elif row_num == '(7)':
                detail_data['Notice_Number'] = cells[3].get_text(strip=True)
            
            # (8) Volcano Location - PENTING untuk koordinat
            elif row_num == '(8)':
                location_text = cells[3].get_text(strip=True)
                detail_data['Volcano_Location'] = location_text
                
                # Parse koordinat: "S 03 deg 31 min 12 sec E 102 deg 37 min 12 sec"
                lat_match = re.search(r'([NS])\s*(\d+)\s*deg\s*(\d+)\s*min\s*(\d+)\s*sec', location_text)
                lon_match = re.search(r'([EW])\s*(\d+)\s*deg\s*(\d+)\s*min\s*(\d+)\s*sec', location_text.split('E')[1] if 'E' in location_text else location_text.split('W')[1] if 'W' in location_text else '')
                
                if lat_match:
                    lat_deg = int(lat_match.group(2))
                    lat_min = int(lat_match.group(3))
                    lat_sec = int(lat_match.group(4))
                    lat = lat_deg + lat_min/60 + lat_sec/3600
                    if lat_match.group(1) == 'S':
                        lat = -lat
                    detail_data['Latitude'] = lat
                
                # Parse longitude from full text
                lon_match = re.search(r'([EW])\s*(\d+)\s*deg\s*(\d+)\s*min\s*(\d+)\s*sec', location_text)
                if lon_match:
                    lon_deg = int(lon_match.group(2))
                    lon_min = int(lon_match.group(3))
                    lon_sec = int(lon_match.group(4))
                    lon = lon_deg + lon_min/60 + lon_sec/3600
                    if lon_match.group(1) == 'W':
                        lon = -lon
                    detail_data['Longitude'] = lon
            
            # (9) Area
            elif row_num == '(9)':
                detail_data['Area'] = cells[3].get_text(strip=True)
            
            # (10) Summit Elevation - PENTING untuk kalkulasi ketinggian
            elif row_num == '(10)':
                elevation_text = cells[3].get_text(strip=True)
                # Parse "6246 FT (1952 M)"
                ft_match = re.search(r'(\d+)\s*FT', elevation_text)
                m_match = re.search(r'(\d+)\s*M', elevation_text)
                
                if ft_match:
                    detail_data['Summit_Elevation_FT'] = int(ft_match.group(1))
                if m_match:
                    detail_data['Summit_Elevation_M'] = int(m_match.group(1))
            
            # (11) Volcanic Activity Summary
            elif row_num == '(11)':
                detail_data['Volcanic_Activity_Summary'] = cells[3].get_text(strip=True)
            
            # (12) Volcanic Cloud Height - SANGAT PENTING untuk model AI
            elif row_num == '(12)':
                detail_data['Volcanic_Cloud_Height'] = cells[3].get_text(strip=True)
            
            # (13) Other Volcanic Cloud Information
            elif row_num == '(13)':
                detail_data['Other_Volcanic_Cloud_Info'] = cells[3].get_text(strip=True)
            
            # (14) Remarks - PENTING untuk info cuaca dan arah angin
            elif row_num == '(14)':
                detail_data['Remarks'] = cells[3].get_text(strip=True)
        
        return detail_data
        
    except Exception as e:
        print(f"    Error scraping detail: {e}")
        return None

In [ ]:
def scrape_vona_complete(pages=1, include_details=True):
    """
    Scrape lengkap data VONA termasuk detail dari setiap laporan
    
    Parameters:
    - pages: jumlah halaman yang akan di-scrape
    - include_details: True untuk scrape detail dari setiap link (lebih lambat tapi lengkap)
    """
    base_url = "https://magma.esdm.go.id/v1/vona/"
    all_data = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for page in range(1, pages + 1):
        try:
            print(f"\n{'='*60}")
            print(f"Scraping halaman {page}/{pages}...")
            print(f"{'='*60}")
            
            # Mengambil halaman list
            url = f"{base_url}?page={page}" if page > 1 else base_url
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"  ✗ Gagal mengakses halaman {page}: Status {response.status_code}")
                continue
                
            soup = BeautifulSoup(response.text, 'html.parser')
            reports = soup.find_all('div', class_='timeline-item')
            
            report_count = 0

            for idx, report in enumerate(reports, 1):
                try:
                    # Skip timeline-day items
                    if 'timeline-day' in report.get('class', []):
                        continue
                    
                    # Ekstrak data dari list page
                    timeline_time = report.find('div', class_='timeline-time')
                    if not timeline_time:
                        continue
                    
                    timestamp_elem = timeline_time.find('small')
                    timestamp = timestamp_elem.get_text(strip=True) if timestamp_elem else 'N/A'
                    
                    alert_button = timeline_time.find('a', class_='btn')
                    alert_level = alert_button.get_text(strip=True) if alert_button else 'N/A'
                    
                    timeline_body = report.find('div', class_='timeline-body')
                    if not timeline_body:
                        continue
                    
                    title_elem = timeline_body.find('p', class_='timeline-title')
                    title = title_elem.get_text(strip=True) if title_elem else 'N/A'
                    
                    author_elem = timeline_body.find('p', class_='timeline-author')
                    author_info = author_elem.get_text(strip=True) if author_elem else 'N/A'
                    
                    text_elem = timeline_body.find('p', class_='timeline-text')
                    description = text_elem.get_text(strip=True) if text_elem else 'N/A'
                    
                    link_elem = timeline_body.find('a', class_='card-link')
                    detail_link = link_elem.get('href') if link_elem else 'N/A'
                    
                    # Data dasar
                    data_row = {
                        "Timestamp": timestamp,
                        "Alert_Level": alert_level,
                        "Volcano_Report": title,
                        "Author_Observatory": author_info,
                        "Description": description,
                        "Detail_Link": detail_link,
                        "Page": page
                    }
                    
                    # Scrape detail jika diminta
                    if include_details and detail_link != 'N/A':
                        print(f"  [{idx}] Scraping detail: {title}...", end=' ')
                        detail_data = scrape_vona_detail(detail_link, headers)
                        
                        if detail_data:
                            data_row.update(detail_data)
                            print("✓")
                        else:
                            print("✗")
                        
                        # Delay untuk menghindari rate limiting
                        time.sleep(0.5)
                    
                    all_data.append(data_row)
                    report_count += 1
                    
                except Exception as e:
                    print(f"  ✗ Error parsing report {idx}: {e}")
                    continue
            
            print(f"\n  ✓ Halaman {page} selesai. Data terkumpul: {report_count} records")
            print(f"  Total akumulasi: {len(all_data)} records")
            
            # Delay antar halaman
            if page < pages:
                time.sleep(2)
                
        except Exception as e:
            print(f"  ✗ Error pada halaman {page}: {e}")
            continue

    if not all_data:
        print("\n⚠ Tidak ada data yang berhasil di-scrape!")
        return pd.DataFrame()
    
    print(f"\n{'='*60}")
    print(f"✓ SCRAPING SELESAI!")
    print(f"{'='*60}")
    print(f"Total data: {len(all_data)} records dari {pages} halaman")
    
    return pd.DataFrame(all_data)

In [ ]:
# SCRAPING LENGKAP dengan Detail (untuk Model AI)
# Set include_details=True untuk mendapat data lengkap
# Set include_details=False untuk scraping cepat tanpa detail

try:
    print("🌋 Memulai scraping data VONA lengkap...")
    print("⏱️  Ini akan memakan waktu karena scraping detail setiap laporan\n")
    
    # Scrape 1 halaman pertama dengan detail lengkap (testing)
    df_vona_complete = scrape_vona_complete(pages=1, include_details=True)
    
    if not df_vona_complete.empty:
        print(f"\n📊 Preview data yang berhasil di-scrape:")
        print(f"Jumlah kolom: {len(df_vona_complete.columns)}")
        print(f"Kolom: {list(df_vona_complete.columns)}\n")
        
        # Tampilkan beberapa kolom penting
        important_cols = ['Volcano_Name', 'Alert_Level', 'Volcanic_Cloud_Height', 
                         'Latitude', 'Longitude', 'Summit_Elevation_M']
        available_cols = [col for col in important_cols if col in df_vona_complete.columns]
        
        if available_cols:
            print(df_vona_complete[available_cols].head())
        else:
            print(df_vona_complete.head())
    else:
        print("\n✗ DataFrame kosong, scraping gagal")
        
except Exception as e:
    print(f"\n✗ Error utama: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Simpan data ke CSV
# Otomatis memilih dataframe yang tersedia

if 'df_vona_complete' in locals() and not df_vona_complete.empty:
    # Prioritas: Simpan data lengkap jika ada
    filename = "data/vona_reports_complete.csv"
    df_vona_complete.to_csv(filename, index=False, encoding='utf-8')
    print(f"✓ Data LENGKAP berhasil disimpan ke: {filename}")
    print(f"  Total records: {len(df_vona_complete)}")
    print(f"  Total kolom: {len(df_vona_complete.columns)}")
    print(f"\n📊 Kolom yang tersimpan:")
    for col in df_vona_complete.columns:
        print(f"  - {col}")
    
elif 'df_vona' in locals() and not df_vona.empty:
    # Fallback: Simpan data basic jika tidak ada data lengkap
    filename = "data/vona_reports_basic.csv"
    df_vona.to_csv(filename, index=False, encoding='utf-8')
    print(f"✓ Data BASIC berhasil disimpan ke: {filename}")
    print(f"  Total records: {len(df_vona)}")
    print(f"  Total kolom: {len(df_vona.columns)}")
    
else:
    print("✗ Tidak ada data untuk disimpan")
    print("💡 Jalankan cell scraping terlebih dahulu!")

In [ ]:
# Analisis & Statistik Data yang Sudah Discrape

if 'df_vona_complete' in locals() and not df_vona_complete.empty:
    df = df_vona_complete
    print("📊 STATISTIK DATA LENGKAP")
elif 'df_vona' in locals() and not df_vona.empty:
    df = df_vona
    print("📊 STATISTIK DATA BASIC")
else:
    print("⚠️ Tidak ada data. Jalankan cell scraping dulu!")
    df = None

if df is not None:
    print("="*60)
    print(f"\n✓ Total Records: {len(df)}")
    print(f"✓ Total Kolom: {len(df.columns)}\n")
    
    # Statistik per Alert Level
    if 'Alert_Level' in df.columns:
        print("🚦 Distribusi Alert Level:")
        print(df['Alert_Level'].value_counts())
        print()
    
    # Statistik per Gunung (jika ada)
    if 'Volcano_Name' in df.columns:
        print("🌋 Top 10 Gunung Paling Aktif (berdasarkan jumlah laporan):")
        print(df['Volcano_Name'].value_counts().head(10))
        print()
    
    # Info data kosong/missing
    print("📋 Missing Data per Kolom:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_pct
    })
    print(missing_df[missing_df['Missing Count'] > 0])
    
    # Contoh data lengkap
    print(f"\n{'='*60}")
    print("📄 Contoh Data (3 baris pertama):")
    print(f"{'='*60}")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    print(df.head(3))